In [1]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import numpy as np
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_rows", 210)

load_dotenv(find_dotenv())

engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host="localhost",
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME"),
))

df_raw = pd.read_sql("SELECT * FROM application_train", engine)

In [2]:
df_raw.duplicated().sum()

np.int64(0)

In [3]:
num = df_raw.select_dtypes("number").drop(columns=['sk_id_curr', 'target'])
med = num.median().replace(0, np.nan)

summary = pd.DataFrame({
    "nunique":         num.nunique(),
    "null_pct":        num.isna().mean().round(3),
    "min":             num.min(),
    "median":          num.median(),
    "max":             num.max(),
    "max_over_median": (num.max() / med).round(1),
})

summary

,nunique,null_pct,min,median,max,max_over_median
cnt_children,15,0.0000,0.0000,0.0000,19.0000,NaN
amt_income_total,2548,0.0000,"25,650.0000","147,150.0000","117,000,000.0000",795.1000
amt_credit,5603,0.0000,"45,000.0000","513,531.0000","4,050,000.0000",7.9000
amt_annuity,13672,0.0000,"1,615.5000","24,903.0000","258,025.5000",10.4000
amt_goods_price,1002,0.0010,"40,500.0000","450,000.0000","4,050,000.0000",9.0000
region_population_relative,81,0.0000,0.0003,0.0188,0.0725,3.8000
days_birth,17460,0.0000,"-25,229.0000","-15,750.0000","-7,489.0000",0.5000
days_employed,12574,0.0000,"-17,912.0000","-1,213.0000","365,243.0000",-301.1000
days_registration,15688,0.0000,"-24,672.0000","-4,504.0000",0.0000,-0.0000
days_id_publish,6168,0.0000,"-7,197.0000","-3,254.0000",0.0000,-0.0000


In [4]:
df_raw['name_contract_type'].value_counts()

name_contract_type
Cash loans         278232
Revolving loans     29279
Name: count, dtype: int64

In [5]:
df_raw['code_gender'].value_counts()

code_gender
F      202448
M      105059
XNA         4
Name: count, dtype: int64

In [6]:
df_raw['flag_own_car'].value_counts()

flag_own_car
N    202924
Y    104587
Name: count, dtype: int64

In [7]:
df_raw['flag_own_realty'].value_counts()

flag_own_realty
Y    213312
N     94199
Name: count, dtype: int64

In [8]:
df_raw['cnt_children'].value_counts()

cnt_children
0     215371
1      61119
2      26749
3       3717
4        429
5         84
6         21
7          7
14         3
8          2
9          2
12         2
10         2
19         2
11         1
Name: count, dtype: int64

In [9]:
odd = df_raw.nlargest(5, 'amt_income_total')
odd[['sk_id_curr', 'amt_income_total', 'amt_credit', 'amt_annuity',
     'name_income_type', 'occupation_type', 'organization_type',
     'name_education_type', 'target']]

,sk_id_curr,amt_income_total,amt_credit,amt_annuity,name_income_type,occupation_type,organization_type,name_education_type,target
12943,114967,"117,000,000.0000","562,491.0000","26,194.5000",Working,Laborers,Business Entity Type 3,Secondary / secondary special,1
203786,336147,"18,000,090.0000","675,000.0000","69,295.5000",Commercial associate,None,Business Entity Type 3,Secondary / secondary special,0
246955,385674,"13,500,000.0000","1,400,503.5000","130,945.5000",Commercial associate,None,Business Entity Type 3,Higher education,0
77837,190160,"9,000,000.0000","1,431,531.0000","132,601.5000",Working,Managers,Business Entity Type 1,Higher education,0
131199,252084,"6,750,000.0000","790,830.0000","52,978.5000",Working,Laborers,Transport: type 4,Higher education,0


In [10]:
df_raw.groupby(pd.qcut(df_raw['amt_goods_price'], 10),observed=True)['target'].agg(['mean', 'count'])

,mean,count
amt_goods_price,,
"(40499.999, 180000.0]",0.0729,39484
"(180000.0, 225000.0]",0.0891,31970
"(225000.0, 270000.0]",0.0773,27990
"(270000.0, 373500.0]",0.1006,23483
"(373500.0, 450000.0]",0.1327,35052
"(450000.0, 522000.0]",0.0654,26475
"(522000.0, 675000.0]",0.0846,42417
"(675000.0, 814500.0]",0.0560,18932
"(814500.0, 1093500.0]",0.0654,30745


In [11]:
df_raw['days_birth'].describe()

count   307,511.0000
mean    -16,036.9951
std       4,363.9886
min     -25,229.0000
25%     -19,682.0000
50%     -15,750.0000
75%     -12,413.0000
max      -7,489.0000
Name: days_birth, dtype: float64

In [12]:
df_raw['days_employed'].describe()

count   307,511.0000
mean     63,815.0459
std     141,275.7665
min     -17,912.0000
25%      -2,760.0000
50%      -1,213.0000
75%        -289.0000
max     365,243.0000
Name: days_employed, dtype: float64

In [13]:
df_raw[df_raw['days_employed'] == 365243]['name_income_type'].value_counts(normalize=True)

name_income_type
Pensioner    0.9996
Unemployed   0.0004
Name: proportion, dtype: float64

In [14]:
df_raw['days_registration'].describe()

count   307,511.0000
mean     -4,986.1203
std       3,522.8863
min     -24,672.0000
25%      -7,479.5000
50%      -4,504.0000
75%      -2,010.0000
max           0.0000
Name: days_registration, dtype: float64

In [15]:
df_raw['own_car_age'].describe()

count   104,582.0000
mean         12.0611
std          11.9448
min           0.0000
25%           5.0000
50%           9.0000
75%          15.0000
max          91.0000
Name: own_car_age, dtype: float64

In [16]:
print(df_raw['own_car_age'].isna().sum())
print((df_raw['flag_own_car'] == 'N').sum())

202929
202924


In [17]:
df_raw.nlargest(10, 'own_car_age')[['sk_id_curr','own_car_age','amt_credit','target']]

,sk_id_curr,own_car_age,amt_credit,target
271817,415025,91.0000,"675,000.0000",1
286598,440757,91.0000,"180,000.0000",0
161501,287098,69.0000,"1,125,000.0000",0
87,100100,65.0000,"796,396.5000",0
136,100156,65.0000,"945,000.0000",0
278,100315,65.0000,"288,873.0000",0
890,100995,65.0000,"450,000.0000",0
896,101004,65.0000,"1,215,000.0000",0
1531,101753,65.0000,"389,484.0000",0
1579,101807,65.0000,"358,213.5000",0


In [18]:
df_raw['own_car_age'].value_counts().sort_index(ascending=False).head(15)

own_car_age
91.0000       2
69.0000       1
65.0000     891
64.0000    2443
63.0000       2
57.0000       1
56.0000       1
55.0000       4
54.0000      12
52.0000       1
51.0000       3
50.0000       1
49.0000       6
48.0000       1
47.0000       1
Name: count, dtype: int64

In [19]:
df_raw['flag_mobil'].value_counts()

flag_mobil
1    307510
0         1
Name: count, dtype: int64

In [20]:
print(df_raw['occupation_type'].value_counts())
print(df_raw['occupation_type'].isnull().value_counts())
print(df_raw[df_raw['occupation_type'].isna()]['name_income_type'].value_counts())

occupation_type
Laborers                 55186
Sales staff              32102
Core staff               27570
Managers                 21371
Drivers                  18603
High skill tech staff    11380
Accountants               9813
Medicine staff            8537
Security staff            6721
Cooking staff             5946
Cleaning staff            4653
Private service staff     2652
Low-skill Laborers        2093
Waiters/barmen staff      1348
Secretaries               1305
Realty agents              751
HR staff                   563
IT staff                   526
Name: count, dtype: int64
occupation_type
False    211120
True      96391
Name: count, dtype: int64
name_income_type
Pensioner               55357
Working                 24920
Commercial associate    12297
State servant            3787
Unemployed                 22
Student                     5
Businessman                 2
Maternity leave             1
Name: count, dtype: int64


In [21]:
mask = df_raw['occupation_type'].isna()
pens = df_raw['name_income_type'] == 'Pensioner'

df_raw.groupby([mask, pens])['target'].agg(['mean','count'])

mean   count
occupation_type name_income_type               
False           False            0.0879  211115
                True             0.0000       5
True            False            0.0803   41034
                True             0.0539   55357

In [22]:
df_raw['cnt_fam_members'].value_counts()

cnt_fam_members
2.0000     158357
1.0000      67847
3.0000      52601
4.0000      24697
5.0000       3478
6.0000        408
7.0000         81
8.0000         20
9.0000          6
10.0000         3
14.0000         2
12.0000         2
20.0000         2
16.0000         2
13.0000         1
15.0000         1
11.0000         1
Name: count, dtype: int64

In [23]:
df_raw[['cnt_children','cnt_fam_members']].corr()

,cnt_children,cnt_fam_members
cnt_children,1.0000,0.8792
cnt_fam_members,0.8792,1.0000


In [24]:
adults = (df_raw['cnt_fam_members'] - df_raw['cnt_children']).rename('adults')
df_raw.groupby(adults)['target'].agg(['mean','count'])

,mean,count
adults,,
1.0000,0.0863,81302
2.0000,0.0787,226207


In [25]:
df_raw.groupby('name_family_status')['target'].agg(['mean','count'])

,mean,count
name_family_status,,
Civil marriage,0.0994,29775
Married,0.0756,196432
Separated,0.0819,19770
Single / not married,0.0981,45444
Unknown,0.0000,2
Widow,0.0582,16088


In [26]:
df_raw['region_rating_client'].value_counts()

region_rating_client
2    226984
3     48330
1     32197
Name: count, dtype: int64

In [27]:
df_raw[['region_rating_client','region_rating_client_w_city']].corr()

,region_rating_client,region_rating_client_w_city
region_rating_client,1.0000,0.9508
region_rating_client_w_city,0.9508,1.0000


In [28]:
print(df_raw['organization_type'].value_counts())
print((df_raw.loc[df_raw['days_employed'] == 365243, 'organization_type'] == 'XNA').sum())

organization_type
Business Entity Type 3    67992
XNA                       55374
Self-employed             38412
Other                     16683
Medicine                  11193
Business Entity Type 2    10553
Government                10404
School                     8893
Trade: type 7              7831
Kindergarten               6880
Construction               6721
Business Entity Type 1     5984
Transport: type 4          5398
Trade: type 3              3492
Industry: type 9           3368
Industry: type 3           3278
Security                   3247
Housing                    2958
Industry: type 11          2704
Military                   2634
Bank                       2507
Agriculture                2454
Police                     2341
Transport: type 2          2204
Postal                     2157
Security Ministries        1974
Trade: type 2              1900
Restaurant                 1811
Services                   1575
University                 1327
Industry: type 7      

In [29]:
concepts = [c[:-4] for c in df_raw.columns if c.endswith('_avg')]
for c in concepts:
    r = df_raw[[f'{c}_avg', f'{c}_mode', f'{c}_medi']].corr().iloc[0, 1:]
    print(f"{c:30s} avg~mode {r.iloc[0]:.3f}   avg~medi {r.iloc[1]:.3f}")

apartments                     avg~mode 0.973   avg~medi 0.995
basementarea                   avg~mode 0.973   avg~medi 0.994
years_beginexpluatation        avg~mode 0.972   avg~medi 0.994
years_build                    avg~mode 0.989   avg~medi 0.998
commonarea                     avg~mode 0.977   avg~medi 0.996
elevators                      avg~mode 0.979   avg~medi 0.996
entrances                      avg~mode 0.978   avg~medi 0.997
floorsmax                      avg~mode 0.986   avg~medi 0.997
floorsmin                      avg~mode 0.986   avg~medi 0.997
landarea                       avg~mode 0.974   avg~medi 0.992
livingapartments               avg~mode 0.970   avg~medi 0.994
livingarea                     avg~mode 0.972   avg~medi 0.996
nonlivingapartments            avg~mode 0.969   avg~medi 0.991
nonlivingarea                  avg~mode 0.966   avg~medi 0.990


In [30]:
print(df_raw['obs_30_cnt_social_circle'].value_counts())
df_raw[df_raw['obs_30_cnt_social_circle'] > 40][
    ['sk_id_curr', 'obs_30_cnt_social_circle', 'obs_60_cnt_social_circle',
     'def_30_cnt_social_circle', 'def_60_cnt_social_circle', 'target']]

obs_30_cnt_social_circle
0.0000      163910
1.0000       48783
2.0000       29808
3.0000       20322
4.0000       14143
5.0000        9553
6.0000        6453
7.0000        4390
8.0000        2967
9.0000        2003
10.0000       1376
11.0000        852
12.0000        652
13.0000        411
14.0000        258
15.0000        166
16.0000        133
17.0000         88
18.0000         46
19.0000         44
20.0000         30
21.0000         29
22.0000         22
23.0000         15
25.0000         11
24.0000         11
27.0000          5
26.0000          3
30.0000          2
28.0000          1
29.0000          1
47.0000          1
348.0000         1
Name: count, dtype: int64


,sk_id_curr,obs_30_cnt_social_circle,obs_60_cnt_social_circle,def_30_cnt_social_circle,def_60_cnt_social_circle,target
77561,189856,47.0000,47.0000,0.0000,0.0000,0
148418,272071,348.0000,344.0000,34.0000,24.0000,0


In [31]:
doc_cols = [c for c in df_raw.columns if c.startswith('flag_document')]
df_raw[doc_cols].sum().sort_values()

flag_document_12         2
flag_document_10         7
flag_document_2         13
flag_document_4         25
flag_document_7         59
flag_document_17        82
flag_document_21       103
flag_document_20       156
flag_document_19       183
flag_document_15       372
flag_document_14       903
flag_document_13      1084
flag_document_9       1198
flag_document_11      1203
flag_document_18      2500
flag_document_16      3053
flag_document_5       4648
flag_document_8      25024
flag_document_6      27078
flag_document_3     218340
dtype: int64

In [32]:
print(df_raw['amt_req_credit_bureau_qrt'].value_counts())
df_raw.loc[df_raw['amt_req_credit_bureau_qrt'] > 20,
    ['sk_id_curr', 'amt_req_credit_bureau_hour', 'amt_req_credit_bureau_day',
     'amt_req_credit_bureau_week', 'amt_req_credit_bureau_mon',
     'amt_req_credit_bureau_qrt', 'amt_req_credit_bureau_year', 'target']]

amt_req_credit_bureau_qrt
0.0000      215417
1.0000       33862
2.0000       14412
3.0000        1717
4.0000         476
5.0000          64
6.0000          28
8.0000           7
7.0000           7
261.0000         1
19.0000          1
Name: count, dtype: int64


,sk_id_curr,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year,target
239591,377322,0.0000,0.0000,0.0000,1.0000,261.0000,0.0000,0


In [33]:
for c in ['fondkapremont_mode', 'housetype_mode', 'wallsmaterial_mode', 'emergencystate_mode']:
    print(df_raw[c].value_counts(dropna=False), '\n')

fondkapremont_mode
None                     210295
reg oper account          73830
reg oper spec account     12080
not specified              5687
org spec account           5619
Name: count, dtype: int64 

housetype_mode
None                154297
block of flats      150503
specific housing      1499
terraced house        1212
Name: count, dtype: int64 

wallsmaterial_mode
None            156341
Panel            66040
Stone, brick     64815
Block             9253
Wooden            5362
Mixed             2296
Monolithic        1779
Others            1625
Name: count, dtype: int64 

emergencystate_mode
No      159428
None    145755
Yes       2328
Name: count, dtype: int64 

